https://business.yelp.com/data/resources/open-dataset/

In [ ]:
import tarfile
import os
import json
from openai import OpenAI
import pandas as pd

In [8]:
tar_filepath = "data/yelp_dataset.tar"
json_path = "data/json"

In [9]:
with tarfile.open(tar_filepath, "r") as tar:
    members = tar.getnames()
    for member in members:
        print(f"Found member: {member}")

    print("\n--- Extracting files... ---")
    # Extract all contents to the specified directory
    tar.extractall(path=json_path)

print(f"\nSuccessfully extracted yelp_dataset into '{json_path}'")

Found member: Dataset_User_Agreement.pdf
Found member: yelp_academic_dataset_business.json
Found member: yelp_academic_dataset_checkin.json
Found member: yelp_academic_dataset_review.json
Found member: yelp_academic_dataset_tip.json
Found member: yelp_academic_dataset_user.json

--- Extracting files... ---

Successfully extracted yelp_dataset into 'data/json'


In [10]:
def read_ndjson_file(filepath):
    """
    Reads a large JSON Lines (NDJSON) file, treating each line as a separate 
    JSON object and returning them as a list of dictionaries.
    """
    print(f"Attempting to read data from: {os.path.abspath(filepath)}")

    all_records = []
    line_count = 0
    error_count = 0
    
    try:
        # Use 'r' for reading text, and iterate line by line
        with open(filepath, 'r', encoding='utf-8') as file:
            for line in file:
                line_count += 1
                # Strip whitespace/newlines from the line
                stripped_line = line.strip()
                
                if not stripped_line:
                    continue # Skip empty lines

                try:
                    # Use json.loads() because we are loading a string (the single line)
                    record = json.loads(stripped_line)
                    all_records.append(record)
                except json.JSONDecodeError as e:
                    print(f"\nSkipping malformed record on Line {line_count}:")
                    print(f"   Error details: {e}")
                    error_count += 1

        if all_records:
            print("\nSuccessfully processed JSON Lines data!")
        else:
            print("\nNo records were successfully loaded.")
            
    except FileNotFoundError:
        print(f"\nError: The file '{filepath}' was not found.")
    except Exception as e:
        print(f"\nAn unexpected error occurred during file reading: {e}")

    print(f"\n--- Summary ---")
    print(f"Total lines processed: {line_count}")
    print(f"Successfully loaded records: {len(all_records)}")
    print(f"Records skipped due to errors: {error_count}")
    
    return all_records

# Specify the path to your JSON Lines file
json_file_path = 'data/json/yelp_academic_dataset_business.json'

# Execute the function and store results in a list of dictionaries
businesses = read_ndjson_file(json_file_path)

Attempting to read data from: /var/home/jvazquez/Documents/Python/YELP-dataset/data/json/yelp_academic_dataset_business.json

Successfully processed JSON Lines data!

--- Summary ---
Total lines processed: 150346
Successfully loaded records: 150346
Records skipped due to errors: 0


In [16]:
# Initialize the client pointing directly to your local LM Studio REST API
client = OpenAI(   
    base_url="http://127.0.0.1:1234/v1", # The exact port from your screenshot
    api_key="lm-studio"                  # API key is required by the library but ignored
)

model_name = "qwen/qwen3-1.7b"

messages = [
    {
        "role": "user", 
        "content": "This is a test prompt just to see if you work! Identify yourself!"
    }
]

# Generate the response
response = client.chat.completions.create(
    model=model_name,
    messages=messages
)

# Print the output
print(response.choices[0].message.content)



Hello! I am an AI assistant designed to help with various tasks and provide information. If you have any questions or need assistance, feel free to ask! 😊


In [ ]:
# def message(_name, _categories):
#     return [
#         {
#             "role": "system", 
#             "content": (
#                 "You are a highly accurate, specialized content classifier. "
#                 "Your sole task is to determine if the user-provided text contains any categories or themes related to restaurants (e.g., dining, food service, cuisine, establishments). "
#                 "You MUST respond with a single character string: '1' if the content IS related to a restaurant, and '0' if it is NOT related."
#                 "However, the category \"Food\" only doesn't mean the business is a restaurant."
#                 "STRICTLY adhere to this format. Do NOT include any surrounding text, explanation, markdown (like ```json```), newlines, spaces, or characters other than the single digit '1' or '0'."
        
#             )
#         },
#         {
#             "role": "user", 
#             "content": f'Analyze the following text: {_name} - [{_categories}]'
#         }
#     ]

In [ ]:
# --- CONFIGURATION ---
BATCH_SIZE = 20  # Number of businesses per LLM request
model_name = "qwen3-1.7b" # Adjust to your LM Studio model ID

def get_batch_message(batch_list):
    """
    Creates a single prompt for a list of businesses to reduce API overhead.
    """
    content_lines = []
    for idx, b in enumerate(batch_list):
        content_lines.append(f"{idx}: Name: {b['name']} | Categories: {b.get('categories', 'None')}")
    
    prompt_text = "\n".join(content_lines)
    
    return [
        {
            "role": "system", 
            "content": (
                "You are a classification bot. For each business, decide if it is PRIMARILY a restaurant or dining establishment. "
                "Retail stores (Target, Walmart), shoe stores, or religious centers are NOT restaurants, even if they sell snacks. "
                "Respond ONLY with a comma-separated list of digits (1 for restaurant, 0 for not). "
                "Example response for 3 items: 1,0,1"
            )
        },
        {
            "role": "user", 
            "content": f"Classify these {len(batch_list)} businesses:\n{prompt_text}"
        }
    ]

# --- PRE-FILTER LOGIC ---
def quick_classify(business):
    cats = str(business.get("categories", "")).lower()
    name = str(business.get("name", "")).lower()
    
    # 1. Obvious Non-Restaurants
    blacklist = ["shoes", "clothing", "synagogues", "churches", "department stores", "jewelry", "automotive"]
    if any(word in cats for word in blacklist):
        return 0
    
    # 2. Obvious Restaurants
    if "restaurants" in cats:
        return 1
        
    # 3. The "Gray Area" (Has 'Food' or 'Cafe' but lacks 'Restaurants' tag)
    if "food" in cats or "cafe" in cats or "coffee" in cats:
        return "LLM_NEEDED"
    
    return 0

# --- MAIN EXECUTION ---
is_business_restaurant = [None] * len(businesses)
needs_llm_indices = []

# Step 1: Pre-filter
print("Step 1: Pre-filtering obvious cases...")
for i, b in enumerate(businesses):
    decision = quick_classify(b)
    if decision != "LLM_NEEDED":
        is_business_restaurant[i] = decision
    else:
        needs_llm_indices.append(i)

print(f"Pre-filter complete. {len(needs_llm_indices)} businesses require LLM analysis.")

# Step 2: Batch Processing for 'LLM_NEEDED' items
print(f"Step 2: Processing batches with {model_name}...")
for i in range(0, len(needs_llm_indices), BATCH_SIZE):
    batch_indices = needs_llm_indices[i : i + BATCH_SIZE]
    batch_data = [businesses[idx] for idx in batch_indices]
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=get_batch_message(batch_data),
            temperature=0,
            max_tokens=50 # We only need a few digits and commas
        )
        
        raw_result = response.choices[0].message.content.strip()
        # Clean potential markdown and split by comma
        clean_result = raw_result.replace(" ", "").split(",")
        
        # Assign results back to the main list
        for sub_idx, val in enumerate(clean_result):
            if sub_idx < len(batch_indices):
                actual_idx = batch_indices[sub_idx]
                is_business_restaurant[actual_idx] = int(val) if val in ['0', '1'] else 0
                
        print(f"Processed batch {i//BATCH_SIZE + 1}/{(len(needs_llm_indices)//BATCH_SIZE)+1}")
        
    except Exception as e:
        print(f"Error in batch {i}: {e}")
        # Fallback to 0 if the LLM fails
        for idx in batch_indices:
            if is_business_restaurant[idx] is None:
                is_business_restaurant[idx] = 0

print("Classification complete.")

Step 1: Pre-filtering obvious cases...
Pre-filter complete. 11467 businesses require LLM analysis.
Step 2: Processing batches with qwen3-1.7b...
Processed batch 1/574
Processed batch 2/574
Processed batch 3/574
Processed batch 4/574
Processed batch 5/574
Processed batch 6/574
Processed batch 7/574
Processed batch 8/574
Processed batch 9/574
Processed batch 10/574
Processed batch 11/574
Processed batch 12/574
Processed batch 13/574
Processed batch 14/574
Processed batch 15/574
Processed batch 16/574
Processed batch 17/574
Processed batch 18/574
Processed batch 19/574
Processed batch 20/574
Processed batch 21/574
Processed batch 22/574
Processed batch 23/574
Processed batch 24/574
Processed batch 25/574
Processed batch 26/574
Processed batch 27/574
Processed batch 28/574
Processed batch 29/574
Processed batch 30/574
Processed batch 31/574
Processed batch 32/574
Processed batch 33/574
Processed batch 34/574
Processed batch 35/574
Processed batch 36/574
Processed batch 37/574
Processed bat

In [ ]:
restaurants_id = []

for index, result in enumerate(is_business_restaurant):
    if result == 1:
        restaurants_id.append(businesses[index].get("business_id"))


data = {
    "restaurants_id" : restaurants_id
}

df = pd.DataFrame(data)

restaurants_csv_output = 'data/csv/restaurants_id.csv'

df.to_csv(restaurants_csv_output, index=True)  # index included by default
